# Library & Data import

In [22]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

In [23]:
df = pd.read_csv('..\Data\\Org_W_Duplicates_data_1_0.csv', low_memory=False)
main_df = pd.read_csv('..\Data\\311_2025_Jan_Dec.csv', low_memory=False) # The dataframe before the changes we applied in FinalTable notebook

<>:1: SyntaxWarning: invalid escape sequence '\D'
<>:2: SyntaxWarning: invalid escape sequence '\D'
<>:1: SyntaxWarning: invalid escape sequence '\D'
<>:2: SyntaxWarning: invalid escape sequence '\D'
C:\Users\Orange\AppData\Local\Temp\ipykernel_23168\466393794.py:1: SyntaxWarning: invalid escape sequence '\D'
  df = pd.read_csv('..\Data\\Org_W_Duplicates_data_1_0.csv', low_memory=False)
C:\Users\Orange\AppData\Local\Temp\ipykernel_23168\466393794.py:2: SyntaxWarning: invalid escape sequence '\D'
  main_df = pd.read_csv('..\Data\\311_2025_Jan_Dec.csv', low_memory=False) # The dataframe before the changes we applied in FinalTable notebook


In [24]:
pd.set_option('display.max_rows', 200)  # In case rows are being cut off 

# First look at the data

In [25]:
# Dataset shape
print("There are {} rows and {} columns in the dataset".format(df.shape[0], df.shape[1]))

There are 3645995 rows and 10 columns in the dataset


In [26]:
# Quick look into the first 5 rows of the dataset
df.head() 

,City,Incident Zip,Agency,Location Type,Closed Date,Created Date,Status,Location,Resolution Time,Complaint_Type
0,WOODSIDE,11377,NYPD,Street/Sidewalk,2025-12-31 02:44:50,2025-12-31 00:30:26,Closed,POINT (-73.907196579197 40.743018746122),0.093333,Vehicles & Parking
1,FAR ROCKAWAY,11694,HPD,RESIDENTIAL BUILDING,2026-01-02 17:18:55,2025-12-31 00:30:19,Closed,POINT (-73.839021963252 40.577671995863),2.700417,Housing & Building Maintenance
2,NEW YORK,10033,HPD,RESIDENTIAL BUILDING,2026-01-02 10:07:10,2025-12-31 00:30:06,Closed,POINT (-73.937239002314 40.851013911318),2.400741,Housing & Building Maintenance
3,BROOKLYN,11207,HPD,RESIDENTIAL BUILDING,2026-01-02 13:49:56,2025-12-31 00:30:05,Closed,POINT (-73.899924654626 40.663886523056),2.555451,Housing & Building Maintenance
4,BROOKLYN,11234,DSNY,Street,2025-12-31 07:32:00,2025-12-31 00:30:00,Closed,POINT (-73.912414463066 40.625294042041),0.293056,Vehicles & Parking


In [27]:
# See each column and its data type
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3645995 entries, 0 to 3645994
Data columns (total 10 columns):
 #   Column           Dtype  
---  ------           -----  
 0   City             object 
 1   Incident Zip     object 
 2   Agency           object 
 3   Location Type    object 
 4   Closed Date      object 
 5   Created Date     object 
 6   Status           object 
 7   Location         object 
 8   Resolution Time  float64
 9   Complaint_Type   object 
dtypes: float64(1), object(9)
memory usage: 278.2+ MB


----=====Valid values=====----
city: Cities in new york metropolitan
Incident Zip: zip code of incident location ( Dtype is str but represents a number - will be converted later)
Agency: Service responsible for the complaint
Location Type: ( Street, Sidewalk, Residential building... )
Status: ( Closed, Open, Processing... )
Location: Cordinates of the complaint
Resolution Time: Time it took to resolve the issue
Complaint Type: ("Housing & Building Maintenance", "Vehicles & Parking"... )

In [28]:
# Lets see the problems people complain about
print(df["Complaint_Type"].unique())

['Vehicles & Parking' 'Housing & Building Maintenance' 'Noise'
 'Sanitation & Trash' 'Street & Infrastructure' 'Public Order & Police'
 'Health & Environmental Safety' 'Other/Misc' 'Trees & Parks'
 'Construction & DOB' 'Taxi & For-Hire Vehicles' 'Animals & Pets']


In [29]:
df["Complaint_Type"].value_counts()

Complaint_Type
Vehicles & Parking                877554
Noise                             831176
Housing & Building Maintenance    786282
Street & Infrastructure           229255
Sanitation & Trash                224815
Public Order & Police             179686
Other/Misc                        177515
Health & Environmental Safety     141139
Trees & Parks                      71259
Construction & DOB                 63241
Animals & Pets                     34471
Taxi & For-Hire Vehicles           29602
Name: count, dtype: int64

# Duplicates 

During data remodeling, we have found out that there is a column called "Additional Details" that causes us to have duplicates. The problem with the column is that for the same complaint, we got more rows, but why?  because each row had a different additional detail. In our clustering problem we want to focus on just the complaint type without more details so we decided to remove any duplicates that appear in our data.

In [30]:
# Amount of duplicates
print(df.duplicated().sum())

203147


In [31]:
# The duplicate rows
df[df.duplicated(keep=False)]


,City,Incident Zip,Agency,Location Type,Closed Date,Created Date,Status,Location,Resolution Time,Complaint_Type
51,NEW YORK,10075,HPD,RESIDENTIAL BUILDING,2026-02-28 06:07:17,2025-12-31 00:23:35,Closed,POINT (-73.957188128108 40.772507525524),59.238681,Housing & Building Maintenance
52,NEW YORK,10075,HPD,RESIDENTIAL BUILDING,2026-02-28 06:07:17,2025-12-31 00:23:35,Closed,POINT (-73.957188128108 40.772507525524),59.238681,Housing & Building Maintenance
53,NEW YORK,10075,HPD,RESIDENTIAL BUILDING,2026-02-28 06:07:17,2025-12-31 00:23:35,Closed,POINT (-73.957188128108 40.772507525524),59.238681,Housing & Building Maintenance
169,BROOKLYN,11226,HPD,RESIDENTIAL BUILDING,2026-01-17 16:13:52,2025-12-31 00:03:42,Closed,POINT (-73.95992971344 40.644328077072),17.673727,Housing & Building Maintenance
170,BROOKLYN,11226,HPD,RESIDENTIAL BUILDING,2026-01-17 16:13:52,2025-12-31 00:03:42,Closed,POINT (-73.95992971344 40.644328077072),17.673727,Housing & Building Maintenance
...,...,...,...,...,...,...,...,...,...,...
3644882,REGO PARK,11374,HPD,RESIDENTIAL BUILDING,2025-05-01 07:10:39,2025-01-01 02:25:08,Closed,POINT (-73.859885543389 40.729233399834),120.198275,Housing & Building Maintenance
3644883,REGO PARK,11374,HPD,RESIDENTIAL BUILDING,2025-05-01 07:10:39,2025-01-01 02:25:08,Closed,POINT (-73.859885543389 40.729233399834),120.198275,Housing & Building Maintenance
3645366,BRONX,10453,HPD,RESIDENTIAL BUILDING,2025-01-28 12:04:14,2025-01-01 01:31:26,Closed,POINT (-73.915556577321 40.846743038691),27.439444,Housing & Building Maintenance
3645367,BRONX,10453,HPD,RESIDENTIAL BUILDING,2025-01-28 12:04:14,2025-01-01 01:31:26,Closed,POINT (-73.915556577321 40.846743038691),27.439444,Housing & Building Maintenance


In [32]:
# Get the index of duplicated rows in df
dup_index = df[df.duplicated()].index
# Use that index to display the full rows from main_df
main_df.loc[dup_index]

,Unique Key,Created Date,Closed Date,Agency,Agency Name,Problem (formerly Complaint Type),Problem Detail (formerly Descriptor),Additional Details,Location Type,Incident Zip,...,Vehicle Type,Taxi Company Borough,Taxi Pick Up Location,Bridge Highway Name,Bridge Highway Direction,Road Ramp,Bridge Highway Segment,Latitude,Longitude,Location
52,67348640,12/31/2025 12:23:35 AM,02/28/2026 06:07:17 AM,HPD,Department of Housing Preservation and Develop...,UNSANITARY CONDITION,MOLD,NaN,RESIDENTIAL BUILDING,10075,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.772508,-73.957188,POINT (-73.957188128108 40.772507525524)
53,67350742,12/31/2025 12:23:35 AM,02/28/2026 06:07:17 AM,HPD,Department of Housing Preservation and Develop...,GENERAL,CABINET,DAMAGED OR MISSING,RESIDENTIAL BUILDING,10075,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.772508,-73.957188,POINT (-73.957188128108 40.772507525524)
170,67348611,12/31/2025 12:03:42 AM,01/17/2026 04:13:52 PM,HPD,Department of Housing Preservation and Develop...,PLUMBING,RADIATOR,BROKEN OR MISSING,RESIDENTIAL BUILDING,11226,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.644328,-73.959930,POINT (-73.95992971344 40.644328077072)
171,67347487,12/31/2025 12:03:42 AM,01/17/2026 04:13:52 PM,HPD,Department of Housing Preservation and Develop...,DOOR/WINDOW,WINDOW FRAME,LOOSE OR DEFECTIVE,RESIDENTIAL BUILDING,11226,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.644328,-73.959930,POINT (-73.95992971344 40.644328077072)
172,67348610,12/31/2025 12:03:42 AM,01/17/2026 04:13:52 PM,HPD,Department of Housing Preservation and Develop...,PLUMBING,RADIATOR,AIR VALVE BROKEN OR MISSING,RESIDENTIAL BUILDING,11226,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.644328,-73.959930,POINT (-73.95992971344 40.644328077072)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3643610,63595060,01/01/2025 05:33:03 AM,01/03/2025 11:25:03 AM,HPD,Department of Housing Preservation and Develop...,APPLIANCE,ELECTRIC/GAS RANGE,NO GAS OR ELECTRICITY,RESIDENTIAL BUILDING,10027,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.806707,-73.949992,POINT (-73.949992233832 40.806706730552)
3644061,63593914,01/01/2025 04:16:54 AM,01/02/2025 02:14:16 PM,HPD,Department of Housing Preservation and Develop...,HEAT/HOT WATER,APARTMENT ONLY,NO HEAT,RESIDENTIAL BUILDING,11429,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.709782,-73.738919,POINT (-73.73891909692 40.70978181729)
3644883,63595195,01/01/2025 02:25:08 AM,05/01/2025 07:10:39 AM,HPD,Department of Housing Preservation and Develop...,UNSANITARY CONDITION,PESTS,MICE,RESIDENTIAL BUILDING,11374,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.729233,-73.859886,POINT (-73.859885543389 40.729233399834)
3645367,63595055,01/01/2025 01:31:26 AM,01/28/2025 12:04:14 PM,HPD,Department of Housing Preservation and Develop...,GENERAL,BELL/BUZZER/INTERCOM,BROKEN OR MISSING,RESIDENTIAL BUILDING,10453,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.846743,-73.915557,POINT (-73.915556577321 40.846743038691)


Let's focus on a specific duplicate with id 170 and 172

In [33]:
main_df.loc[[170,172]]

,Unique Key,Created Date,Closed Date,Agency,Agency Name,Problem (formerly Complaint Type),Problem Detail (formerly Descriptor),Additional Details,Location Type,Incident Zip,...,Vehicle Type,Taxi Company Borough,Taxi Pick Up Location,Bridge Highway Name,Bridge Highway Direction,Road Ramp,Bridge Highway Segment,Latitude,Longitude,Location
170,67348611,12/31/2025 12:03:42 AM,01/17/2026 04:13:52 PM,HPD,Department of Housing Preservation and Develop...,PLUMBING,RADIATOR,BROKEN OR MISSING,RESIDENTIAL BUILDING,11226,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.644328,-73.95993,POINT (-73.95992971344 40.644328077072)
172,67348610,12/31/2025 12:03:42 AM,01/17/2026 04:13:52 PM,HPD,Department of Housing Preservation and Develop...,PLUMBING,RADIATOR,AIR VALVE BROKEN OR MISSING,RESIDENTIAL BUILDING,11226,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.644328,-73.95993,POINT (-73.95992971344 40.644328077072)


As we can see, the rows are almost identical because its the same complaint: "Plumbing" but why did it create 2 rows? Because of "Additional Details" column! We have got 2 different details for the same problem. As we said in the beginning we will remove all duplicates. ( 363809 rows which is 10% of the data)

In [34]:
# Dropping duplicates
df = df.drop_duplicates(keep=False)
print(df.duplicated().sum())

0


In [35]:
# Let's save the table as csv
df.to_csv("Org_data_2_0.csv", index=False)

# Missing Values

In [36]:
# Number of missing value for each feature.
df.isnull().sum()

City               155630
Incident Zip        29771
Agency                  0
Location Type      427166
Closed Date         69637
Created Date            0
Status                  0
Location            49862
Resolution Time     69637
Complaint_Type          0
dtype: int64

In [37]:
# Percentage of missing values for each feature.
df.isnull().mean()*100

City                4.650629
Incident Zip        0.889635
Agency              0.000000
Location Type      12.764831
Closed Date         2.080935
Created Date        0.000000
Status              0.000000
Location            1.490006
Resolution Time     2.080935
Complaint_Type      0.000000
dtype: float64

As we can see, most of the columns have missing values. We will decide on how to deal with them later on.

This is the data which is Null/None/NaN... But what if the missing data was written as Unknown or Unavailable? Let's check the most common encodings.